In [7]:
import sys, os
sys.path.insert(0, "../src")
sys.path.insert(0, "../applications/bodyfat")

import numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from scipy.spatial import Delaunay

import load_bodyfat
import bodyfat_explainer as bfx
from load_bodyfat import FEATS, TARGET, COLS
FEATS, TARGET

(['abdomen', 'hip', 'chest'], 'bodyfat')

In [8]:
df = load_bodyfat.load()
print(f"{len(df)} real rows loaded")
df[COLS].corr().round(2)

251 real rows loaded


,abdomen,hip,chest,bodyfat
abdomen,1.00,0.87,0.91,0.81
hip,0.87,1.00,0.83,0.62
chest,0.91,0.83,1.00,0.70
bodyfat,0.81,0.62,0.70,1.00


In [9]:
sub = (df[COLS] - df[COLS].mean()) / df[COLS].std()
bb = LinearRegression().fit(sub[FEATS], sub[TARGET])
print(f"black-box R^2: {bb.score(sub[FEATS], sub[TARGET]):.3f}")
pca = PCA(2); xy = pca.fit_transform(sub[COLS])
print(f"2D embedding keeps {pca.explained_variance_ratio_.sum():.1%} of variance")

black-box R^2: 0.695
2D embedding keeps 94.6% of variance


In [10]:
K = 8
corner_idx = bfx.farthest_point_corners(xy, K)
tri = Delaunay(xy[corner_idx])
cvals = df[TARGET].values[corner_idx]

for j, gi in enumerate(corner_idx):
    print(f"P{j}: '{bfx.describe_corner(df, gi, FEATS)}'  bodyfat~{df[TARGET].values[gi]:.0f}%")

P0: 'low hip'  bodyfat~20%
P1: 'high abdomen, high hip, high chest'  bodyfat~35%
P2: 'high abdomen, high hip, high chest'  bodyfat~48%
P3: 'low abdomen, low hip, low chest'  bodyfat~4%
P4: 'high abdomen, high hip, high chest'  bodyfat~24%
P5: 'high abdomen, high hip, high chest'  bodyfat~34%
P6: 'high hip'  bodyfat~8%
P7: 'high abdomen'  bodyfat~33%


In [11]:
for i in (3, 40, 120, 200):
    if i < len(df):
        print(bfx.explain(i, df, xy, tri, corner_idx, cvals, FEATS)[0]); print()

Person 3 (measurements: abdomen=86, hip=101, chest=102):
  strongly resembles the 'high hip' build (bodyfat~8%) - 61%
  moderately resembles the 'low hip' build (bodyfat~20%) - 27%
  slightly resembles the 'low abdomen, low hip, low chest' build (bodyfat~4%) - 12%
  => estimated body fat = 10.5%  (measured: 10.4%)

Person 40 (measurements: abdomen=126, hip=126, chest=128):
  very strongly resembles the 'high abdomen, high hip, high chest' build (bodyfat~34%) - 100%
  => estimated body fat = 34.5%  (measured: 34.5%)

Person 120 (measurements: abdomen=99, hip=104, chest=104):
  strongly resembles the 'high abdomen' build (bodyfat~33%) - 72%
  slightly resembles the 'high hip' build (bodyfat~8%) - 17%
  slightly resembles the 'high abdomen, high hip, high chest' build (bodyfat~24%) - 11%
  => estimated body fat = 27.5%  (measured: 27.9%)

Person 200 (measurements: abdomen=86, hip=97, chest=91):
  unlike any prototype on record - no honest explanation available



In [12]:
conf = []
for i in range(len(df)):
    s = tri.find_simplex(xy[i])
    if s < 0: continue
    w = bfx.bary(xy[i], tri, s); w = np.clip(w, 0, None); w = w / w.sum()
    conf.append((w.max(), i))
conf.sort()

print("MOST BLENDED (uncertain):")
print(bfx.explain(conf[0][1], df, xy, tri, corner_idx, cvals, FEATS)[0])
print("\nMOST SINGLE-PROTOTYPE (confident):")
print(bfx.explain(conf[-1][1], df, xy, tri, corner_idx, cvals, FEATS)[0])

MOST BLENDED (uncertain):
Person 50 (measurements: abdomen=87, hip=98, chest=90):
  moderately resembles the 'low abdomen, low hip, low chest' build (bodyfat~4%) - 34%
  moderately resembles the 'low hip' build (bodyfat~20%) - 33%
  moderately resembles the 'high hip' build (bodyfat~8%) - 32%
  => estimated body fat = 10.4%  (measured: 10.2%)

MOST SINGLE-PROTOTYPE (confident):
Person 214 (measurements: abdomen=122, hip=113, chest=120):
  very strongly resembles the 'high abdomen, high hip, high chest' build (bodyfat~48%) - 100%
  => estimated body fat = 47.5%  (measured: 47.5%)


In [13]:
outside = [i for i in range(len(df)) if tri.find_simplex(xy[i]) < 0]
coverage = 1 - len(outside) / len(df)
print(f"coverage: {coverage:.1%}  ({len(outside)} people abstained on)")
if outside:
    i = outside[0]
    print(f"\nexample abstention — Person {i}:")
    print(bfx.explain(i, df, xy, tri, corner_idx, cvals, FEATS)[0])

coverage: 84.1%  (40 people abstained on)

example abstention — Person 2:
Person 2 (measurements: abdomen=88, hip=99, chest=96):
  unlike any prototype on record - no honest explanation available
